In [19]:
import warnings
from config import *
import os
import re
import pandas as pd
import numpy as np

# Show all rows
pd.set_option('display.max_rows', None)
# Show all columns
pd.set_option('display.max_columns', None)
# Do not truncate column values
pd.set_option('display.max_colwidth', None)

In [20]:
randhrs = "data/original data/randhrs1992_2020v2.dta"
hrs_tracker = "data/original data/trk2022tr_r.dta"

In [21]:
# Set wave e.g:
# year = 2 * (wave - 5) from 2002 to 2016

wave = 14
year = 2 * (wave - 5)

In [22]:
chunks_main = pd.read_stata(randhrs)

In [23]:
chunks_main[f'r{wave}imrc']=chunks_main[f'r{wave}imrcw'].add(chunks_main[f'r{wave}imrcp'],fill_value=0)
chunks_main[f'r{wave}ser7']=chunks_main[f'r{wave}ser7w'].add(chunks_main[f'r{wave}ser7p'],fill_value=0)
chunks_main[f'r{wave}dlrc']=chunks_main[f'r{wave}dlrcw'].add(chunks_main[f'r{wave}dlrcp'],fill_value=0)

In [24]:
w = chunks_main[f'r{wave}bwc20w']
p = chunks_main[f'r{wave}bwc20p']

# 1. figure out the full set of categories
all_cats = w.cat.categories.union(p.cat.categories)

# 2. combine: take w if present, else p; result is an Index of values (with NaN where both are missing)
combined = w.fillna(p)

# 3. turn into a Categorical with the full category list
chunks_main[f'r{wave}bwc20'] = pd.Categorical(combined, categories=all_cats)

# quick check
print(chunks_main[f'r{wave}bwc20'].dtype)       # should show Categorical
print(chunks_main[f'r{wave}bwc20'].head(10))    # sample values, including NaN where both were missing


category
0                   NaN
1                   NaN
2                   NaN
3                   NaN
4                   NaN
5                   NaN
6    1.correct, 2nd try
7                   NaN
8    2.correct, 1st try
9                   NaN
Name: r14bwc20, dtype: category
Categories (3, object): ['0.incorrect' < '1.correct, 2nd try' < '2.correct, 1st try']


In [25]:
# Time-invariant variables (from base interview)
timeinvariant_household_col = ['hidpn']
timeinvariant_respondent_col = ['gender', 'edyrs', 'bplace']
timeinvariant_col = [f'h{col}' for col in timeinvariant_household_col] + \
                    [f'ra{col}' for col in timeinvariant_respondent_col]

# Wave-specific variables for respondent and household
wave_household_col = ['child'] # Number of children

wave_respondent_col = [
    'lbrf',     # Labor force status
    'shlt',     # Self-rated health
    'agey_m',   #age
    'height',
    'weight',
    'smokev',   # Smoker status
    'proxy',    # Proxy interview indicator
    'imrc',  # immediate recall of a list of 10 words
    'dlrc',  # delayed recall of a list of 10 words
    'ser7',  # five trials of serial 7s
    'bwc20',  # backward counting
    'prmem',  # proxy rating of respondent memory
    'iadl5a',  # iadl5a = sum(phonea, moneya, medsa, shopa, mealsa), referred as "iadlza" in appendix
    'cendiv', # Census Division
    'mstat', # Marital Status
    'effort', # CESD Everything an effort
    'hibpe', # Ever had high blood pressure
    'diabe', # Ever had diabetes
    'cancre', # Ever had cancer
    'lunge', # Ever had lung disease
    'hearte', # Ever had heart problems
    'stroke', # Ever had stroke
    'arthre', # Ever had arthritis
    'slfmem', # Self-rated memory
    'livpar', # Number of living parents
    'momage', # Mother age current/at death
    'dadage', # Father age current/at death
    'livsib', # Number of living siblings
    'hlthlm', # Health problems limit work
    'hosp',	# Hospital stay(Overnight hospital stay since PrvIvw before death)
    'nrshom', # Nursing home stay (whether the Respondent reports any overnight nursing home stay in the reference period)
    'nrstim', # Nursing home stays
    'nrsnit', # Nights in nursing home (number of nights over all stays)
    'doctim', # Doctor visits (reported number of visits)
    'depres', # CESD Felt depressed
    'sleepr', # CESD Sleep was restless
    'whappy', # CESD Was happy
    'flone',    # CESD Felt lonely
    'fsad',     # CESD Felt sad
    'going',    # CESD Could not get going
    'enlife',   # CESD Enjoyed life
    'drink',    # Ever drinks any alcohol
    'smoken',   # Smokes now
    'toilta',   # Some Difficulty-Using the toilet
    'adl5a',    # Some Difficulty-IADLs /0-3
    'mapa',     # Some Difficulty-Use a map
    'walksa',   # Some Difficulty-Walk sev blocks
    'walk1a',   # Some Difficulty-Walk one block
    'sita',     # Some Difficulty-Sit for 2 hours
    'chaira',   # Some Difficulty-Get up fr chair
    'climsa',   # Some Difficulty-Clmb sev flt str
    'clim1a',   # Some Difficulty-Clmb 1 flt stair
    'stoopa',   # Some Difficulty-Stoop/Kneel/Crch
    'lifta',    # Some Difficulty-Lift/carry 10 lbs
    'dimea',    # Some Difficulty-Pick up a dime
    'armsa',    # Some Difficulty-Rch/xtnd arms up
    'pusha',    # Some Difficulty-Push/pull lg obj
    'mobila',   # Some Difficulty-Mobility /0-5
    'lgmusa',   # Some Diff-Large Muscle /0-4
    'grossa',   # Walk1/R,Clim1,Bed,Bath/0-5
    'finea',    # Dime/Eat/Dress /0-3
]

if wave==6:
    wave_respondent_col.append('vigact') # (For w6): Whether vigorous phys act 3+/wk
else:
    wave_respondent_col.append('vgactx') # (For w7-w15) Whether vigorous phys act 3+/wk, Freq vigorous phys active {finer scale}

# Construct final columns to load
columns = timeinvariant_col + \
          [f'h{wave}{col}' for col in wave_household_col] + \
          [f'r{wave}{col}' for col in wave_respondent_col]

# Read the selected columns from the dataset
chunks_main = chunks_main[columns]


# Map the correct 'prfin' column based on year
col_prfin = {2:'HA011',
             4:'JA011',
             6:'KA011',
             8:'LA011',
             10:'MA011',
             12: 'NA011',
             14: 'OA011',
             16: 'PA011',
             18: 'QA011',
             20: 'RA011',
             }
original_prfin_col = col_prfin[year]
new_prfin_col = f'r{wave}prfin'

# Load prefin column dataset 
prfin_data = pd.read_csv('data/original data/'+f'h{year}A_R.csv')

In [26]:
# Create unique person ID
#prfin_data['hhidpn'] = (prfin_data['HHID'] + prfin_data['PN']).astype(int)
prfin_data['hhidpn']=(prfin_data['HHID'].astype(str)+'0'+prfin_data['PN'].astype(str)).astype(int)
prfin_data.drop(['HHID','PN'],axis=1,inplace=True)

# Recode original prfin values into categories
prfin_data[new_prfin_col] = pd.cut(
    prfin_data[original_prfin_col],
    bins=[0, 1, 2, 3],
    labels=['1.none', '2.some', '3.prevented'],
    ordered=True
)

# Keep only hhidpn and recoded prfin variable
prfin_data = prfin_data[['hhidpn', new_prfin_col]]

# Merge with main dataset on hhidpn
chunks = pd.merge(
    chunks_main,
    prfin_data,
    on='hhidpn',
    how='left'
)

In [27]:
# Define new column name for race/ethnicity based on wave
raceeth_col = f'r{wave}raceeth'

# Load relevant columns from HRS tracker file
raceeth_tracker = pd.read_stata(
    hrs_tracker,
    columns=['HHID', 'PN', 'RACE', 'HISPANIC', 'NIWWAVE']
)

# Generate unique person ID
raceeth_tracker['hhidpn'] = (raceeth_tracker['HHID'] + raceeth_tracker['PN']).astype(int)
raceeth_tracker.drop(['HHID', 'PN'], axis=1, inplace=True)
# Recode race/ethnicity following HRS coding logic:
# 0 = Non-Hispanic White
# 1 = Non-Hispanic Black
# 2 = Hispanic
# 3 = Non-Hispanic Other
raceeth_tracker.loc[raceeth_tracker['HISPANIC'].isin([1, 2, 3]), raceeth_col] = 2
raceeth_tracker.loc[(raceeth_tracker['HISPANIC'].isin([0, 5])) & (raceeth_tracker['RACE'] == 1), raceeth_col] = 0
raceeth_tracker.loc[(raceeth_tracker['HISPANIC'].isin([0, 5])) & (raceeth_tracker['RACE'] == 2), raceeth_col] = 1
raceeth_tracker.loc[(raceeth_tracker['HISPANIC'].isin([0, 5])) & (raceeth_tracker['RACE'] == 7), raceeth_col] = 3

# Step 2: Filter for rows where NIWWAVE == 1
raceeth_tracker = raceeth_tracker[raceeth_tracker['NIWWAVE'] == 1]

# Drop original identifiers and raw race/ethnicity variables
raceeth_tracker.drop(columns=[ 'HISPANIC', 'RACE'], inplace=True)

# Check distribution of the recoded race/ethnicity variable
raceeth_tracker[raceeth_col].value_counts(dropna=False)


r14raceeth
0.0    14098
1.0     4146
2.0     2911
3.0      722
NaN       16
Name: count, dtype: int64

In [28]:
# Merge race/ethnicity tracker data into the main dataset using `hhidpn` as the key
chunks = pd.merge(
    chunks,                # main analysis dataset (already includes wave, prfin, etc.)
    raceeth_tracker,       # race/ethnicity tracker data
    on='hhidpn',           # merge key
    how='left'             # keep all cases from main dataset
)
chunks.head()

,hhidpn,ragender,raedyrs,rabplace,h14child,r14lbrf,r14shlt,r14agey_m,r14height,r14weight,r14smokev,r14proxy,r14imrc,r14dlrc,r14ser7,r14bwc20,r14prmem,r14iadl5a,r14cendiv,r14mstat,r14effort,r14hibpe,r14diabe,r14cancre,r14lunge,r14hearte,r14stroke,r14arthre,r14slfmem,r14livpar,r14momage,r14dadage,r14livsib,r14hlthlm,r14hosp,r14nrshom,r14nrstim,r14nrsnit,r14doctim,r14depres,r14sleepr,r14whappy,r14flone,r14fsad,r14going,r14enlife,r14drink,r14smoken,r14toilta,r14adl5a,r14mapa,r14walksa,r14walk1a,r14sita,r14chaira,r14climsa,r14clim1a,r14stoopa,r14lifta,r14dimea,r14armsa,r14pusha,r14mobila,r14lgmusa,r14grossa,r14finea,r14vgactx,r14prfin,NIWWAVE,r14raceeth
0,1010,1.male,16.0,2.mid atlantic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010,2.female,8.0,3.en central,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3010,1.male,12.0,9.pacific,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0
3,3020,2.female,16.0,8.mountain,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0
4,10001010,1.male,12.0,2.mid atlantic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0


In [29]:
# Convert time-invariant categorical variables to float (delete float setting)
timeinvariant_categorical_cols = ['ragender', 'raedyrs']

for col in timeinvariant_categorical_cols:
    chunks[col] = chunks[col].cat.codes.astype(float)
    chunks.loc[chunks[col] == -1, col] = float('nan')

# Recode one-hot indicator
for col in [
    'smokev', 'proxy', 'prmem', 'prfin','bwc20',  'effort',
    'hibpe', 'diabe', 'cancre', 'lunge', 'hearte', 'stroke',
    'vgactx',
    'arthre', 'slfmem', 'hlthlm', 'hosp', 'nrshom',
    'depres','sleepr', 'whappy', 'flone', 'fsad', 'going',
    'enlife', 'drink', 'smoken', 'toilta', 'mapa', 'walksa',
    'walk1a', 'sita', 'chaira', 'climsa', 'clim1a', 'stoopa',
    'lifta', 'dimea', 'armsa','pusha'
]:

# delete categorical variables: 'cendiv', 'mstat', 'bplace' due to non-ordinal

    wave_col = f'r{wave}{col}'
    chunks.loc[:, wave_col] = chunks[wave_col].cat.codes.astype(float)
    chunks.loc[chunks[wave_col] == -1, wave_col] = float('nan')

# Bin education into stage groups
raedstg: ['1.0-7', '2.8-11', '3.12', '4.13+']
chunks['raedstg'] = pd.cut(
    chunks['raedyrs'],
    bins=[-1, 7, 11, 12, chunks['raedyrs'].max()],
    labels=['1.0-7', '2.8-11', '3.12', '4.13+']
)

# Convert all wave-specific categorical variables to float
wave_categorical_cols = [
    'lbrf',     # Labor force status
    'shlt',     # Self-rated health
]

for col in wave_categorical_cols:
    wave_col = f'r{wave}{col}'
    chunks[wave_col] = chunks[wave_col].cat.codes.astype(float)
    chunks.loc[chunks[wave_col] == -1, wave_col] = float('nan')

C:\Users\13022\AppData\Local\Temp\ipykernel_29632\3527993302.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1. -1. -1. ... -1. -1.  1.]' has dtype incompatible with category, please explicitly cast to a compatible dtype first.
  chunks.loc[:, wave_col] = chunks[wave_col].cat.codes.astype(float)
C:\Users\13022\AppData\Local\Temp\ipykernel_29632\3527993302.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1. -1. -1. ... -1. -1.  0.]' has dtype incompatible with category, please explicitly cast to a compatible dtype first.
  chunks.loc[:, wave_col] = chunks[wave_col].cat.codes.astype(float)
C:\Users\13022\AppData\Local\Temp\ipykernel_29632\3527993302.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1. -1. -1. ... -1. -1. -1.]' has dtype incompatible with cat

In [30]:
# Drop the original 'raedyrs' column (already binned into 'raedstg' and encoded into 'redleq17')
chunks.drop(columns=['raedyrs'], inplace=True)

# (Optional) Reorder columns: Move key variables like hhidpn and race/ethnicity to the front
cols_order = ['hhidpn']
if f'r{wave}raceeth' in chunks.columns:
    cols_order.append(f'r{wave}raceeth')
if 'ragender' in chunks.columns:
    cols_order.append('ragender')
if 'raedstg' in chunks.columns:
    cols_order.append('raedstg')
if 'redleq17' in chunks.columns:
    cols_order.append('redleq17')

# Add remaining columns after key ones
cols_remaining = [col for col in chunks.columns if col not in cols_order]
chunks = chunks[cols_order + cols_remaining]
chunks[f'r{wave}raceeth'].value_counts(dropna=False)


r14raceeth
NaN    20528
0.0    14098
1.0     4146
2.0     2911
3.0      722
Name: count, dtype: int64

In [31]:
# Define variable names
proxy_col = f'r{wave}proxy'
demscr_col = f'r{wave}demscr'
demcls_col = f'r{wave}demcls'

# Define column groups used to calculate cognitive score
self_dem_cols_wave = [f'r{wave}{col}' for col in SELF_DEM_COLS]
proxy_dem_cols_wave = [f'r{wave}{col}' for col in PROXY_DEM_COLS]

# Recode for self-respondents
self_mask = (chunks[proxy_col] == 0)
print(self_mask.sum())
chunks.loc[self_mask, demscr_col] = chunks.loc[self_mask, self_dem_cols_wave].sum(
    axis=1, min_count=len(self_dem_cols_wave)
)
chunks.loc[self_mask & (chunks[demscr_col] <= 6), demcls_col] = 1.0
chunks.loc[self_mask & (chunks[demscr_col] > 6), demcls_col] = 0.0

# Recode for proxy-respondents
proxy_mask = (chunks[proxy_col] == 1)
chunks.loc[proxy_mask, demscr_col] = chunks.loc[proxy_mask, proxy_dem_cols_wave].sum(
    axis=1, min_count=len(proxy_dem_cols_wave)
)
chunks.loc[proxy_mask & (chunks[demscr_col] >= 6), demcls_col] = 1.0
chunks.loc[proxy_mask & (chunks[demscr_col] < 6), demcls_col] = 0.0

# Drop rows with missing demscr (as they can't be classified)
# chunks.dropna(subset=[demscr_col], inplace=True)
# Drop raw demscr score now that demcls has been derived
chunks.drop(columns=[demscr_col], inplace=True)

16483


In [32]:
def remove_wave_prefix(col, wave):
    # Remove 'ra', 'r11', 'ha', 'h11' prefixes only
    return re.sub(rf'^(r|h)(a|{wave})', '', col)

# Apply clean renaming to all columns
chunks.rename(
    columns={col: remove_wave_prefix(col, wave) for col in chunks.columns},
    inplace=True
)

chunks['raceeth']

0        NaN
1        NaN
2        0.0
3        0.0
4        0.0
5        NaN
6        0.0
7        0.0
8        0.0
9        0.0
10       0.0
11       0.0
12       0.0
13       3.0
14       0.0
15       0.0
16       2.0
17       2.0
18       NaN
19       NaN
20       1.0
21       NaN
22       NaN
23       NaN
24       0.0
25       NaN
26       0.0
27       0.0
28       NaN
29       0.0
30       NaN
31       NaN
32       NaN
33       2.0
34       NaN
35       1.0
36       1.0
37       NaN
38       3.0
39       NaN
40       1.0
41       NaN
42       NaN
43       NaN
44       NaN
45       1.0
46       1.0
47       0.0
48       3.0
49       3.0
50       0.0
51       0.0
52       NaN
53       1.0
54       NaN
55       0.0
56       0.0
57       NaN
58       0.0
59       0.0
60       0.0
61       0.0
62       NaN
63       NaN
64       NaN
65       NaN
66       NaN
67       0.0
68       NaN
69       3.0
70       3.0
71       0.0
72       0.0
73       0.0
74       0.0
75       NaN
76       NaN

In [33]:
core_feature_cols = [
    'hhidpn', 'NIWWAVE', # keys to merge, not x variables
    # all seleted x variables
    'edstg', "child", "lbrf", "shlt", "agey_m", "height", "weight", "smokev", "proxy", "cendiv", "mstat", "effort", "hibpe", "diabe",
    'vgactx',
    "slfmem", "livpar", "momage", "dadage", "livsib", "hlthlm", "hosp", "nrshom", "nrstim", "nrsnit", "doctim", "depres", "sleepr", "whappy", "flone", "fsad", "going", "enlife", "drink", "smoken", "cancre", "lunge", "hearte", "stroke", "arthre", "toilta",
    "adl5a", "mapa", "walksa", "walk1a", "sita", "chaira", "climsa", "clim1a", "stoopa", "lifta", "dimea", "armsa", "pusha",
    "mobila", "lgmusa", "grossa", "finea", "raceeth","gender",
    "demcls" # y: Binary cognitive impairment classification
]
chunks_new=chunks[core_feature_cols]

# Convert the 'raceeth' column to a categorical variable
chunks_new['raceeth'] = chunks_new['raceeth'].astype('category')

C:\Users\13022\AppData\Local\Temp\ipykernel_29632\3866896667.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chunks_new['raceeth'] = chunks_new['raceeth'].astype('category')


In [34]:
chunks_new = pd.get_dummies(chunks_new,columns=chunks_new.select_dtypes(include=['category']).columns.tolist(),drop_first=True)

In [35]:
chunks_new.to_csv('data/preprocessed data/'+f'20{year}.csv',na_rep="NA" ,index=False)

In [36]:
#Select 'demcls' and rename it to '__demcls' and hhidpn
chunks_new.rename(columns={'demcls':f'{year}demcls'}, inplace=True)
y = chunks_new[['hhidpn' , f'{year}demcls']]
y.to_csv('data/preprocessed data/'+f'_{year}y.csv',na_rep="NA" ,index=False)